# NUTRILYS - Étape 7 : fine-tuning de CamemBERT (Google Colab)

Ce notebook entraîne un modèle de langage français sur une tâche « texte seul » de l'étape 6, et l'évalue sur **exactement le même jeu de test** que les modèles de référence.

**Avant de commencer**
1. Menu **Exécution > Modifier le type d'exécution**, choisir **GPU T4**, enregistrer.
2. Avoir sous la main le fichier exporté sur ton PC : `data/processed/colab/nova_name.parquet` (ou `nutriscore_text.parquet`).
3. Lancer les cellules dans l'ordre (**Exécution > Tout exécuter**). Une fenêtre demandera le fichier à charger.

À la fin, deux fichiers sont téléchargés automatiquement :
- `<tâche>_camembert.parquet` : prédictions sur le jeu de test, à placer dans `data/processed/predictions/` ;
- `<tâche>_camembert_metrics.json` : métriques, à placer dans `reports/models/`.

Durée indicative sur GPU T4 avec DistilCamemBERT : 15 à 30 minutes pour `nova_name`, davantage pour `nutriscore_text` (textes plus longs).

## 1. Paramètres

Seule cellule à modifier. Pour la seconde tâche, remplacer `TASK` par `"nutriscore_text"` et relancer tout le notebook.

In [ ]:
TASK = "nova_name"  # "nova_name" ou "nutriscore_text"
MODEL_NAME = "cmarkea/distilcamembert-base"  # plus rapide ; ou "almanach/camembert-base"
MAX_LENGTH = 64 if TASK == "nova_name" else 256  # longueur maximale des textes, en tokens
EPOCHS = 2
BATCH_SIZE = 64
LEARNING_RATE = 5e-5
MAX_TRAIN = None  # par exemple 100_000 pour aller plus vite ; None = tout l'entraînement
SEED = 42
DATA_PATH = f"{TASK}.parquet"

## 2. Environnement

In [ ]:
import json
import math
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

IN_COLAB = "google.colab" in sys.modules
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if IN_COLAB and device.type != "cuda":
    raise RuntimeError(
        "Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4, puis relancer."
    )
print("Appareil :", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Données

In [ ]:
if IN_COLAB and not Path(DATA_PATH).exists():
    from google.colab import files

    print(f"Choisir le fichier {DATA_PATH} exporté depuis le projet.")
    files.upload()

df = pd.read_parquet(DATA_PATH)
classes = sorted(df["target"].unique())  # ordre naturel : 1 à 4, ou a à e
label_of = {c: i for i, c in enumerate(classes)}
df["label"] = df["target"].map(label_of)

train = df[df["split"] == "train"]
valid = df[df["split"] == "valid"]
test = df[df["split"] == "test"]
if MAX_TRAIN and len(train) > MAX_TRAIN:
    train = train.groupby("label", group_keys=False).sample(
        frac=MAX_TRAIN / len(train), random_state=SEED
    )
print("Classes :", ", ".join(str(c) for c in classes))
print(f"Entraînement {len(train)}, validation {len(valid)}, test {len(test)}")
print("Exemple :", train["text"].iloc[0][:200])

## 4. Modèle

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(classes),
    id2label={i: str(c) for i, c in enumerate(classes)},
    label2id={str(c): i for i, c in enumerate(classes)},
    ignore_mismatched_sizes=True,  # la tête de classification est créée pour nos classes
).to(device)
print(f"{MODEL_NAME} : {sum(p.numel() for p in model.parameters()) / 1e6:.0f} M paramètres")

## 5. Entraînement

Boucle d'entraînement PyTorch explicite (pas de `Trainer`), pour rester lisible et stable d'une version de Transformers à l'autre :
- perte pondérée par classe, comme la régression logistique de l'étape 6 (`class_weight="balanced"`) ;
- montée progressive puis décroissance linéaire du taux d'apprentissage ;
- précision mixte sur GPU pour aller plus vite ;
- à chaque époque, F1 macro sur la validation ; on garde la meilleure époque.

In [ ]:
def encode(texts):
    return tokenizer(
        list(texts), truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt"
    ).to(device)


def batches(frame, shuffle):
    order = np.random.permutation(len(frame)) if shuffle else np.arange(len(frame))
    texts, labels = frame["text"].to_numpy(), frame["label"].to_numpy()
    for start in range(0, len(frame), BATCH_SIZE):
        idx = order[start : start + BATCH_SIZE]
        yield texts[idx], torch.as_tensor(labels[idx], device=device)


use_amp = device.type == "cuda"


@torch.no_grad()
def predict_proba(frame):
    model.eval()
    out = []
    for texts, _ in batches(frame, shuffle=False):
        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = model(**encode(texts)).logits
        out.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)


counts = np.bincount(train["label"], minlength=len(classes))
weights = torch.tensor(len(train) / (len(classes) * np.maximum(counts, 1)), dtype=torch.float)
loss_fn = torch.nn.CrossEntropyLoss(weight=weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = EPOCHS * math.ceil(len(train) / BATCH_SIZE)
warmup = max(1, int(0.06 * total_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda step: min((step + 1) / warmup, max(0.0, (total_steps - step) / (total_steps - warmup))),
)
scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

best_f1, best_state, history = -1.0, None, []
start = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    model.train()
    progress = tqdm(
        batches(train, shuffle=True),
        total=math.ceil(len(train) / BATCH_SIZE),
        desc=f"Époque {epoch}/{EPOCHS}",
    )
    for texts, labels in progress:
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            loss = loss_fn(model(**encode(texts)).logits.float(), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        progress.set_postfix(perte=f"{loss.item():.3f}")
    valid_pred = predict_proba(valid).argmax(axis=1)
    valid_f1 = f1_score(valid["label"], valid_pred, average="macro")
    history.append({"epoch": epoch, "valid_f1_macro": round(float(valid_f1), 4)})
    print(f"Époque {epoch} : F1 macro validation {valid_f1:.3f}")
    if valid_f1 > best_f1:
        best_f1 = valid_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
fit_seconds = time.perf_counter() - start
model.load_state_dict(best_state)
print(f"Entraînement : {fit_seconds / 60:.1f} min, meilleure F1 validation {best_f1:.3f}")

## 6. Évaluation sur le jeu de test

Mêmes métriques qu'à l'étape 6, calculées une seule fois, sur le jeu de test.

In [ ]:
t0 = time.perf_counter()
proba = predict_proba(test)
predict_seconds = time.perf_counter() - t0
y_true, y_pred = test["label"].to_numpy(), proba.argmax(axis=1)

metrics = {
    "task": TASK,
    "model": "camembert",
    "model_name": MODEL_NAME,
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
    "within_one": float(np.mean(np.abs(y_true - y_pred) <= 1)),
    "n_test": int(len(test)),
    "n_train": int(len(train)),
    "fit_seconds": round(fit_seconds, 1),
    "predict_ms_per_1000": round(1000 * 1000 * predict_seconds / len(test), 1),
    "sample": 0,
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "epochs": EPOCHS,
    "max_length": MAX_LENGTH,
    "history": history,
}
for key in ("f1_macro", "accuracy", "within_one"):
    print(f"{key:>12} : {metrics[key]:.3f}")

## 7. Enregistrement et téléchargement des résultats

In [ ]:
classes_arr = np.asarray(classes)
predictions = pd.DataFrame(
    {
        "code": test["code"].to_numpy(),
        "y_true": classes_arr[y_true],
        "y_pred": classes_arr[y_pred],
        "confidence": proba.max(axis=1),
    }
)
pred_path = f"{TASK}_camembert.parquet"
metrics_path = f"{TASK}_camembert_metrics.json"
predictions.to_parquet(pred_path, index=False)
Path(metrics_path).write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print("Fichiers écrits :", pred_path, metrics_path)

if IN_COLAB:
    from google.colab import files

    files.download(pred_path)
    files.download(metrics_path)

## 8. Facultatif : conserver le modèle entraîné

Le modèle n'est pas nécessaire pour la comparaison. Pour le garder, décommenter ces lignes : il sera enregistré dans ton Google Drive (environ 270 Mo pour DistilCamemBERT).

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# out = f"/content/drive/MyDrive/nutrilys/{TASK}_camembert"
# model.save_pretrained(out)
# tokenizer.save_pretrained(out)